In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

CONV_SAMPLES = {
    'greetings' : ['Hi', 'hello', 'How are you', 'hey there', 'hey'],
    'taxi'      : ['book a cab', 'need a ride', 'find me a cab'],
    'weather'   : ['what is the weather in tokyo', 'weather germany',
                   'what is the weather like in kochi',
                   'what is the weather like', 'is it hot outside'],
    'datetime'  : ['what day is today', 'todays date', 'what time is it now',
                   'time now', 'what is the time'],
    'music'     : ['play the Beatles', 'shuffle songs', 'make a sound']
}

# Prepare training data
texts = []
labels = []

for label, samples in CONV_SAMPLES.items():
    for s in samples:
        texts.append(s)
        labels.append(label)

# Vectorizer + Classifier
vectorizer = TfidfVectorizer()
clf = LinearSVC()

# Train
X = vectorizer.fit_transform(texts)
clf.fit(X, labels)

# Prediction function
def predict_intent(text):
    x = vectorizer.transform([text])
    return clf.predict(x)[0]

print(predict_intent('will it rain today'))          # weather
print(predict_intent("play playlist rock n'roll"))   # music
print(predict_intent("what's the hour"))             # datetime


datetime
music
datetime


In [3]:
import spacy
from spacy.pipeline import EntityRuler

# Load a blank English model
nlp = spacy.blank("en")

# Add EntityRuler
ruler = nlp.add_pipe("entity_ruler")

# Training samples
X_WEATHER = [
    'what is the weather in tokyo',
    'weather germany',
    'what is the weather like in kochi'
]

Y_WEATHER = [
    {'intent': 'weather', 'place': 'tokyo'},
    {'intent': 'weather', 'place': 'germany'},
    {'intent': 'weather', 'place': 'kochi'}
]

# Build patterns from training data
patterns = []
for text, label in zip(X_WEATHER, Y_WEATHER):
    place = label['place']
    patterns.append({
        "label": "PLACE",
        "pattern": place
    })

ruler.add_patterns(patterns)

# Intent classifier (simple keyword-based)
def extract_entities(text):
    doc = nlp(text)
    entities = {ent.label_: ent.text for ent in doc.ents}
    return entities

print(extract_entities("is it raining in tokyo today"))
print(extract_entities("weather report for germany"))


{'PLACE': 'tokyo'}
{'PLACE': 'germany'}


In [5]:
# EX_WEATHER.predict('what is the weather in London')
extract_entities("weather report for germany")

{'PLACE': 'germany'}

In [6]:
import spacy
from spacy.pipeline import EntityRuler

# Training data
x = [
    'what is the weather in tokyo',
    'what is the weather',
    'what is the weather like in kochi'
]

y = [
    {'intent': 'weather', 'place': 'tokyo'},
    {'intent': 'weather', 'place': 'here'},
    {'intent': 'weather', 'place': 'kochi'}
]

# Build spaCy pipeline
nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler")

# Create patterns from training labels
patterns = []
for text, label in zip(x, y):
    place = label['place']
    if place != "here":   # "here" is not a real location
        patterns.append({"label": "PLACE", "pattern": place})

ruler.add_patterns(patterns)

# Prediction function (Eywa-style)
def predict_entities(text):
    doc = nlp(text)
    
    # Extract place if found
    places = [ent.text for ent in doc.ents if ent.label_ == "PLACE"]
    
    if places:
        place = places[0]
    else:
        place = "here"   # fallback like Eywa
    
    return {
        "intent": "weather",
        "place": place
    }

# Test
x_test = "what is the weather in london like"
print(predict_entities(x_test))


{'intent': 'weather', 'place': 'here'}


In [8]:
import requests
import logging

LOGGER = logging.getLogger("main")

API_KEY = "YOUR_API_KEY"   # replace with your key

def get_weather_forecast(place):
    LOGGER.warning(place)

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": place,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "weather" in data:
        return data["weather"][0]["description"]
    else:
        return "Weather data not available"

print(get_weather_forecast("London"))


London


Weather data not available


In [9]:
X_GREETING = ['Hii', 'helllo', 'Howdy', 'hey there', 'hey', 'Hi']
Y_GREETING = [{'greet': 'Hii'}, {'greet': 'helllo'}, {'greet': 'Howdy'},
              {'greet': 'hey'}, {'greet': 'hey'}, {'greet': 'Hi'}]

EX_GREETING = EntityExtractor()
EX_GREETING.fit(X_GREETING, Y_GREETING)

NameError: name 'EntityExtractor' is not defined

In [9]:
X_DATETIME = ['what day is today', 'date today', 'what time is it now', 'time now']
Y_DATETIME = [{'intent' : 'day', 'target': 'today'}, {'intent' : 'date', 'target': 'today'},
              {'intent' : 'time', 'target': 'now'}, {'intent' : 'time', 'target': 'now'}]

EX_DATETIME = EntityExtractor()
EX_DATETIME.fit(X_DATETIME, Y_DATETIME)

In [10]:
import datetime

_EXTRACTORS = {
  'taxi': None,
  'weather': EX_WEATHER,
  'greetings': EX_GREETING,
  'datetime': EX_DATETIME,
  'music': None
}

def question_and_answer(u_query):
    '''Answer a user query
    '''
    q_class = CLF.predict(u_query)
    if _EXTRACTORS[q_class] is None:
      return 'Sorry, you have to upgrade your software!'

    q_entities = _EXTRACTORS[q_class].predict(u_query)
    if q_class == 'greetings':
      return q_entities.get('greet', 'hello')
    
    if q_class == 'weather':
      place = q_entities.get('place', 'London').replace('_', ' ')
      return 'The forecast for {} is {}'.format(
          place,
          get_weather_forecast(place)
      )

    if q_class == 'datetime':
      return 'Today\'s date is {}'.format(
          datetime.datetime.today().strftime('%B %d, %Y')
      )
    
    return 'I couldn\'t understand what you said. I am sorry.'

In [ ]:
while True:
  query = input('\nHow can I help you? ')
  print(question_and_answer(query))

germany


The forecast for germany is light rain
